### Import

In [1]:
import os
import sys
import re
import numpy as np
import pandas as pd
import datetime as dt
from tqdm import tqdm
from matplotlib import pyplot as plt 

from multiprocessing import Pool
from multiprocessing import cpu_count

pd.set_option('display.max_columns', 500)

### General parameters

In [21]:
path_csv  = "../Data/csvExtract/"
path_timeseries = "../Data/Output"

### List of variables

In [3]:
all_variables = []

with open("../Data/csvExtract/variables.txt", "r") as f:
    for variable in f:
        all_variables.append(variable.strip())
        
len(all_variables)

230

### Read csv files

In [4]:
def dataframe_from_csv(path, header=0, index_col=False):
    return pd.read_csv(path, header=header, index_col=index_col)

In [5]:
def filter_on_variabels(all_tables, all_variables):
    all_tables = all_tables[all_tables['LABEL'].isin(all_variables)]
    return all_tables

### Outlier detection

In [6]:
def outlier_removal(all_series):
    
    column = list(all_series.columns)
    for col in column:
        if col not in ['ICUSTAY_ID', 'CHARTTIME', 'Note', 'Discharge_Note', 'Heart Rhythm', 'Ventilator Mode', 
                       'Ventilator Type', 'Ventilator', 'Sedation Score', 'Ramsey SedationScale', 
                       'EtCO2 Clinical indication']:
            try:
                all_series[col] = all_series[col].astype(float)
            except:
                continue
                
    all_series.loc[all_series['UrineOutput_IO'] < 0 ,     'UrineOutput_IO'] = np.nan
    all_series.loc[all_series['UrineOutput_IO'] > 1200 ,  'UrineOutput_IO'] = np.nan
    all_series.loc[all_series['Stool_IO'] < 0 ,     'Stool_IO'] = np.nan
    all_series.loc[all_series['Stool_IO'] > 1500 ,  'Stool_IO'] = np.nan
    all_series.loc[all_series['Propofol_IO'] < 0 ,     'Propofol_IO'] = np.nan
    all_series.loc[all_series['Propofol_IO'] > 1100 ,  'Propofol_IO'] = np.nan
    all_series.loc[all_series['Fentanyl_IO'] < 0 ,   'Fentanyl_IO'] = np.nan
    all_series.loc[all_series['Fentanyl_IO'] > 8 ,   'Fentanyl_IO'] = np.nan
    all_series.loc[all_series['Insulin_IO'] < 0 ,    'Insulin_IO'] = np.nan
    all_series.loc[all_series['Insulin_IO'] > 150 ,  'Insulin_IO'] = np.nan
    all_series.loc[all_series['Heparin_IO'] < 0 ,      'Heparin_IO'] = np.nan
    all_series.loc[all_series['Heparin_IO'] > 25000 ,  'Heparin_IO'] = np.nan
    all_series.loc[all_series['Midazolam_IO'] < 0 ,    'Midazolam_IO'] = np.nan
    all_series.loc[all_series['Midazolam_IO'] > 120 ,  'Midazolam_IO'] = np.nan
    all_series.loc[all_series['Dexmedetomidine_IO'] < 0 ,     'Dexmedetomidine_IO'] = np.nan
    all_series.loc[all_series['Dexmedetomidine_IO'] > 1100 ,  'Dexmedetomidine_IO'] = np.nan
    all_series.loc[all_series['Albumin_IO'] < 0 ,    'Albumin_IO'] = np.nan
    all_series.loc[all_series['Albumin_IO'] > 600 ,  'Albumin_IO'] = np.nan
    all_series.loc[all_series['Ceftriaxone_IO'] < 0 ,    'Ceftriaxone_IO'] = np.nan
    all_series.loc[all_series['Ceftriaxone_IO'] > 10 ,   'Ceftriaxone_IO'] = np.nan
    all_series.loc[all_series['Cefazolin_IO'] < 0 ,    'Cefazolin_IO'] = np.nan
    all_series.loc[all_series['Cefazolin_IO'] > 10 ,   'Cefazolin_IO'] = np.nan
    all_series.loc[all_series['Cefepime_IO'] < 0 ,    'Cefepime_IO'] = np.nan
    all_series.loc[all_series['Cefepime_IO'] > 10 ,   'Cefepime_IO'] = np.nan
    all_series.loc[all_series['Ceftazidime_IO'] < 0 ,    'Ceftazidime_IO'] = np.nan
    all_series.loc[all_series['Ceftazidime_IO'] > 10 ,   'Ceftazidime_IO'] = np.nan
    all_series.loc[all_series['Vancomycin_IO'] < 0 ,    'Vancomycin_IO'] = np.nan
    all_series.loc[all_series['Vancomycin_IO'] > 10 ,   'Vancomycin_IO'] = np.nan
    all_series.loc[all_series['Clindamycin_IO'] < 0 ,    'Clindamycin_IO'] = np.nan
    all_series.loc[all_series['Clindamycin_IO'] > 10 ,   'Clindamycin_IO'] = np.nan
    all_series.loc[all_series['Metronidazole_IO'] < 0 ,    'Metronidazole_IO'] = np.nan
    all_series.loc[all_series['Metronidazole_IO'] > 10 ,   'Metronidazole_IO'] = np.nan
    all_series.loc[all_series['Meropenem_IO'] < 0 ,    'Meropenem_IO'] = np.nan
    all_series.loc[all_series['Meropenem_IO'] > 10 ,   'Meropenem_IO'] = np.nan
    all_series.loc[all_series['Acyclovir_IO'] < 0 ,    'Acyclovir_IO'] = np.nan
    all_series.loc[all_series['Acyclovir_IO'] > 10 ,   'Acyclovir_IO'] = np.nan
    all_series.loc[all_series['Azithromycin_IO'] < 0 ,    'Azithromycin_IO'] = np.nan
    all_series.loc[all_series['Azithromycin_IO'] > 10 ,   'Azithromycin_IO'] = np.nan
    all_series.loc[all_series['Levofloxacin_IO'] < 0 ,    'Levofloxacin_IO'] = np.nan
    all_series.loc[all_series['Levofloxacin_IO'] > 10 ,   'Levofloxacin_IO'] = np.nan
    all_series.loc[all_series['Micafungin_IO'] < 0 ,    'Micafungin_IO'] = np.nan
    all_series.loc[all_series['Micafungin_IO'] > 10 ,   'Micafungin_IO'] = np.nan
    all_series.loc[all_series['Fluconazole_IO'] < 0 ,    'Fluconazole_IO'] = np.nan
    all_series.loc[all_series['Fluconazole_IO'] > 10 ,   'Fluconazole_IO'] = np.nan
    all_series.loc[all_series['Thiamine_IO'] < 0 ,     'Thiamine_IO'] = np.nan
    all_series.loc[all_series['Thiamine_IO'] > 1000 ,  'Thiamine_IO'] = np.nan
    all_series.loc[all_series['Dobutamine_IO'] < 0 ,    'Dobutamine_IO'] = np.nan
    all_series.loc[all_series['Dobutamine_IO'] > 600 ,  'Dobutamine_IO'] = np.nan
    all_series.loc[all_series['Milrinone_IO'] < 0 ,    'Milrinone_IO'] = np.nan
    all_series.loc[all_series['Milrinone_IO'] > 75 ,   'Milrinone_IO'] = np.nan
    all_series.loc[all_series['Fluids_IO'] < 0 ,     'Fluids_IO'] = np.nan
    all_series.loc[all_series['Fluids_IO'] > 1100 ,  'Fluids_IO'] = np.nan
    all_series.loc[all_series['OralIntake_IO'] < 0 ,    'OralIntake_IO'] = np.nan
    all_series.loc[all_series['OralIntake_IO'] > 600 ,  'OralIntake_IO'] = np.nan
    all_series.loc[all_series['P.O._IO'] < 0 ,     'P.O._IO'] = np.nan
    all_series.loc[all_series['P.O._IO'] > 1000 ,  'P.O._IO'] = np.nan
    all_series.loc[all_series['IVPB_IO'] < 0 ,    'IVPB_IO'] = np.nan
    all_series.loc[all_series['IVPB_IO'] > 200 ,  'IVPB_IO'] = np.nan
    all_series.loc[all_series['Crystalloids_IO'] < 0 ,    'Crystalloids_IO'] = np.nan
    all_series.loc[all_series['Crystalloids_IO'] > 12000 ,  'Crystalloids_IO'] = np.nan
    all_series.loc[all_series['Amiodarone_IO'] < 0 ,    'Amiodarone_IO'] = np.nan
    all_series.loc[all_series['Amiodarone_IO'] > 350 ,  'Amiodarone_IO'] = np.nan
    all_series.loc[all_series['Epinephrine_IO'] < 0 ,   'Epinephrine_IO'] = np.nan
    all_series.loc[all_series['Epinephrine_IO'] > 5 ,   'Epinephrine_IO'] = np.nan
    all_series.loc[all_series['Nicardipine_IO'] < 0 ,    'Nicardipine_IO'] = np.nan
    all_series.loc[all_series['Nicardipine_IO'] > 150 ,  'Nicardipine_IO'] = np.nan
    all_series.loc[all_series['Pantoprazole_IO'] < 0 ,    'Pantoprazole_IO'] = np.nan
    all_series.loc[all_series['Pantoprazole_IO'] > 50 ,   'Pantoprazole_IO'] = np.nan
    all_series.loc[all_series['Diltiazem_IO'] < 0 ,    'Diltiazem_IO'] = np.nan
    all_series.loc[all_series['Diltiazem_IO'] > 110 ,  'Diltiazem_IO'] = np.nan
    all_series.loc[all_series['Nitroglycerin_IO'] < 0 ,    'Nitroglycerin_IO'] = np.nan
    all_series.loc[all_series['Nitroglycerin_IO'] > 120 ,  'Nitroglycerin_IO'] = np.nan
    all_series.loc[all_series['NeuroblockAgent_IO'] < 0 ,    'NeuroblockAgent_IO'] = np.nan
    all_series.loc[all_series['NeuroblockAgent_IO'] > 220 ,  'NeuroblockAgent_IO'] = np.nan
    all_series.loc[all_series['K-IV_IO'] < 0 ,    'K-IV_IO'] = np.nan
    all_series.loc[all_series['K-IV_IO'] > 60 ,   'K-IV_IO'] = np.nan
    all_series.loc[all_series['Ca-IV_IO'] < 0 ,    'Ca-IV_IO'] = np.nan
    all_series.loc[all_series['Ca-IV_IO'] > 500 ,  'Ca-IV_IO'] = np.nan
    all_series.loc[all_series['Ca-nonIV_IO'] < 0 ,    'Ca-nonIV_IO'] = np.nan
    all_series.loc[all_series['Ca-nonIV_IO'] > 30 ,   'Ca-nonIV_IO'] = np.nan
    all_series.loc[all_series['Mg-IV_IO'] < 0 ,    'Mg-IV_IO'] = np.nan
    all_series.loc[all_series['Mg-IV_IO'] > 30 ,   'Mg-IV_IO'] = np.nan
    all_series.loc[all_series['Mg-nonIV_IO'] < 0 ,    'Mg-nonIV_IO'] = np.nan
    all_series.loc[all_series['Mg-nonIV_IO'] > 150 ,  'Mg-nonIV_IO'] = np.nan
    all_series.loc[all_series['P-IV_IO'] < 0 ,    'P-IV_IO'] = np.nan
    all_series.loc[all_series['P-IV_IO'] > 50 ,   'P-IV_IO'] = np.nan
    all_series.loc[all_series['P-nonIV_IO'] < 0 ,    'P-nonIV_IO'] = np.nan
    all_series.loc[all_series['P-nonIV_IO'] > 120 ,  'P-nonIV_IO'] = np.nan
    all_series.loc[all_series['BetaBlockers_IO'] < 0 ,     'BetaBlockers_IO'] = np.nan
    all_series.loc[all_series['BetaBlockers_IO'] > 1200 ,  'BetaBlockers_IO'] = np.nan
    all_series.loc[all_series['CaBlockers_IO'] < 0 ,    'CaBlockers_IO'] = np.nan
    all_series.loc[all_series['CaBlockers_IO'] > 700 ,  'CaBlockers_IO'] = np.nan
    all_series.loc[all_series['LoopDiuretics_IO'] < 0 ,    'LoopDiuretics_IO'] = np.nan
    all_series.loc[all_series['LoopDiuretics_IO'] > 350 ,  'LoopDiuretics_IO'] = np.nan
    all_series.loc[all_series['PNutrition_IO'] < 0 ,     'PNutrition_IO'] = np.nan
    all_series.loc[all_series['PNutrition_IO'] > 2500 ,  'PNutrition_IO'] = np.nan
    all_series.loc[all_series['Dextrose_IO'] < 0 ,     'Dextrose_IO'] = np.nan
    all_series.loc[all_series['Dextrose_IO'] > 1100 ,  'Dextrose_IO'] = np.nan
    all_series.loc[all_series['POnutrition_IO'] < 0 ,     'POnutrition_IO'] = np.nan
    all_series.loc[all_series['POnutrition_IO'] > 1500 ,  'POnutrition_IO'] = np.nan
    all_series.loc[all_series['Vasopressors_IO'] < 0 ,    'Vasopressors_IO'] = np.nan
    all_series.loc[all_series['Vasopressors_IO'] > 250 ,  'Vasopressors_IO'] = np.nan
    all_series.loc[all_series['Temperature'] < 30 ,    'Temperature'] = np.nan
    all_series.loc[all_series['Temperature'] > 45 ,    'Temperature'] = np.nan
    all_series.loc[all_series['PT'] < 0 ,    'PT'] = np.nan
    all_series.loc[all_series['PT'] > 80 ,   'PT'] = np.nan
    all_series.loc[all_series['PTT'] < 0 ,    'PTT'] = np.nan
    all_series.loc[all_series['PTT'] > 200 ,  'PTT'] = np.nan
    all_series.loc[all_series['INR(PT)'] < 0 ,    'INR(PT)'] = np.nan
    all_series.loc[all_series['INR(PT)'] > 10 ,   'INR(PT)'] = np.nan
    all_series.loc[all_series['pH'] < 5 ,   'pH'] = np.nan
    all_series.loc[all_series['pH'] > 9 ,   'pH'] = np.nan
    all_series.loc[all_series['Lactate'] < 0 ,    'Lactate'] = np.nan
    all_series.loc[all_series['Lactate'] > 30 ,   'Lactate'] = np.nan
    all_series.loc[all_series['Lactate Dehydrogenase (LD)'] < 0 ,     'Lactate Dehydrogenase (LD)'] = np.nan
    all_series.loc[all_series['Lactate Dehydrogenase (LD)'] > 3000 ,  'Lactate Dehydrogenase (LD)'] = np.nan
    all_series.loc[all_series['Base Excess'] < -35 ,    'Base Excess'] = np.nan
    all_series.loc[all_series['Base Excess'] > 35 ,     'Base Excess'] = np.nan
    all_series.loc[all_series['Anion Gap'] < 0 ,    'Anion Gap'] = np.nan
    all_series.loc[all_series['Anion Gap'] > 40 ,   'Anion Gap'] = np.nan
    all_series.loc[all_series['Bicarbonate'] < 0 ,    'Bicarbonate'] = np.nan
    all_series.loc[all_series['Bicarbonate'] > 60 ,   'Bicarbonate'] = np.nan
    all_series.loc[all_series['Creatinine'] < 0 ,    'Creatinine'] = np.nan
    all_series.loc[all_series['Creatinine'] > 30 ,   'Creatinine'] = np.nan
    all_series.loc[all_series['Hematocrit'] < 0 ,    'Hematocrit'] = np.nan
    all_series.loc[all_series['Hematocrit'] > 75 ,   'Hematocrit'] = np.nan
    all_series.loc[all_series['Hemoglobin'] < 0 ,    'Hemoglobin'] = np.nan
    all_series.loc[all_series['Hemoglobin'] > 50 ,   'Hemoglobin'] = np.nan
    all_series.loc[all_series['Bilirubin, Total'] < 0 ,    'Bilirubin, Total'] = np.nan
    all_series.loc[all_series['Bilirubin, Total'] > 50 ,   'Bilirubin, Total'] = np.nan
    all_series.loc[all_series['Bilirubin, Direct'] < 0 ,    'Bilirubin, Direct'] = np.nan
    all_series.loc[all_series['Bilirubin, Direct'] > 50 ,   'Bilirubin, Direct'] = np.nan
    all_series.loc[all_series['Bilirubin, Indirect'] < 0 ,    'Bilirubin, Indirect'] = np.nan
    all_series.loc[all_series['Bilirubin, Indirect'] > 50 ,   'Bilirubin, Indirect'] = np.nan
    all_series.loc[all_series['BUN'] < 0 ,    'BUN'] = np.nan
    all_series.loc[all_series['BUN'] > 170 ,  'BUN'] = np.nan
    all_series.loc[all_series['MCV'] < 10 ,    'MCV'] = np.nan
    all_series.loc[all_series['MCV'] > 150 ,   'MCV'] = np.nan
    all_series.loc[all_series['MCH'] < 10 ,    'MCH'] = np.nan
    all_series.loc[all_series['MCH'] > 50 ,    'MCH'] = np.nan
    all_series.loc[all_series['MCHC'] < 10 ,    'MCHC'] = np.nan
    all_series.loc[all_series['MCHC'] > 50 ,    'MCHC'] = np.nan
    all_series.loc[all_series['RDW'] < 5 ,    'RDW'] = np.nan
    all_series.loc[all_series['RDW'] > 40 ,   'RDW'] = np.nan
    all_series.loc[all_series['RBC'] < 0 ,    'RBC'] = np.nan
    all_series.loc[all_series['RBC'] > 250 ,  'RBC'] = np.nan
    all_series.loc[all_series['WBC'] < 0 ,    'WBC'] = np.nan
    all_series.loc[all_series['WBC'] > 100 ,  'WBC'] = np.nan
    all_series.loc[all_series['Red Blood Cells'] < 0 ,    'Red Blood Cells'] = np.nan
    all_series.loc[all_series['Red Blood Cells'] > 10 ,   'Red Blood Cells'] = np.nan
    all_series.loc[all_series['White Blood Cells'] < 0 ,    'White Blood Cells'] = np.nan
    all_series.loc[all_series['White Blood Cells'] > 70 ,   'White Blood Cells'] = np.nan
    all_series.loc[all_series['Platelet Count'] < 0 ,     'Platelet Count'] = np.nan
    all_series.loc[all_series['Platelet Count'] > 1000 ,  'Platelet Count'] = np.nan
    all_series.loc[all_series['Glucose'] < 0 ,    'Glucose'] = np.nan
    all_series.loc[all_series['Glucose'] > 900 ,  'Glucose'] = np.nan
    all_series.loc[all_series['Ammonia'] < 0 ,    'Ammonia'] = np.nan
    all_series.loc[all_series['Ammonia'] > 400 ,  'Ammonia'] = np.nan
    all_series.loc[all_series['Magnesium'] < 0 ,    'Magnesium'] = np.nan
    all_series.loc[all_series['Magnesium'] > 10 ,   'Magnesium'] = np.nan
    all_series.loc[all_series['Phosphate'] < 0 ,    'Phosphate'] = np.nan
    all_series.loc[all_series['Phosphate'] > 15 ,   'Phosphate'] = np.nan
    all_series.loc[all_series['Alkaline Phosphatase'] < 0 ,    'Alkaline Phosphatase'] = np.nan
    all_series.loc[all_series['Alkaline Phosphatase'] > 700 ,  'Alkaline Phosphatase'] = np.nan
    all_series.loc[all_series['Potassium'] < 0 ,    'Potassium'] = np.nan
    all_series.loc[all_series['Potassium'] > 10 ,   'Potassium'] = np.nan
    all_series.loc[all_series['Sodium'] < 60 ,    'Sodium'] = np.nan
    all_series.loc[all_series['Sodium'] > 200 ,   'Sodium'] = np.nan
    all_series.loc[all_series['Chloride'] < 20 ,    'Chloride'] = np.nan
    all_series.loc[all_series['Chloride'] > 180 ,   'Chloride'] = np.nan
    all_series.loc[all_series['Ionized Calcium'] < 0 ,    'Ionized Calcium'] = np.nan
    all_series.loc[all_series['Ionized Calcium'] > 9 ,    'Ionized Calcium'] = np.nan
    all_series.loc[all_series['Calcium, Total'] < 0 ,    'Calcium, Total'] = np.nan
    all_series.loc[all_series['Calcium, Total'] > 20 ,   'Calcium, Total'] = np.nan
    all_series.loc[all_series['Cholesterol, Total'] < 0 ,    'Cholesterol, Total'] = np.nan
    all_series.loc[all_series['Cholesterol, Total'] > 500 ,  'Cholesterol, Total'] = np.nan
    all_series.loc[all_series['Cholesterol, HDL'] < 0 ,    'Cholesterol, HDL'] = np.nan
    all_series.loc[all_series['Cholesterol, HDL'] > 200 ,  'Cholesterol, HDL'] = np.nan
    all_series.loc[all_series['Cholesterol, LDL'] < 0 ,    'Cholesterol, LDL'] = np.nan
    all_series.loc[all_series['Cholesterol, LDL'] > 400 ,  'Cholesterol, LDL'] = np.nan
    all_series.loc[all_series['C-Reactive Protein (CRP)'] < 0 ,    'C-Reactive Protein (CRP)'] = np.nan
    all_series.loc[all_series['C-Reactive Protein (CRP)'] > 400 ,  'C-Reactive Protein (CRP)'] = np.nan
    all_series.loc[all_series['pO2'] < 0 ,    'pO2'] = np.nan
    all_series.loc[all_series['pO2'] > 600 ,  'pO2'] = np.nan
    all_series.loc[all_series['pCO2'] < 0 ,    'pCO2'] = np.nan
    all_series.loc[all_series['pCO2'] > 175 ,  'pCO2'] = np.nan
    all_series.loc[all_series['ALT'] < 0 ,     'ALT'] = np.nan
    all_series.loc[all_series['ALT'] > 1200 ,  'ALT'] = np.nan
    all_series.loc[all_series['AST'] < 0 ,     'AST'] = np.nan
    all_series.loc[all_series['AST'] > 1200 ,  'AST'] = np.nan
    all_series.loc[all_series['Amylase'] < 0 ,    'Amylase'] = np.nan
    all_series.loc[all_series['Amylase'] > 800 ,  'Amylase'] = np.nan
    all_series.loc[all_series['Lipase'] < 0 ,    'Lipase'] = np.nan
    all_series.loc[all_series['Lipase'] > 800 ,  'Lipase'] = np.nan
    all_series.loc[all_series['Differential-Eos'] < 0 ,     'Differential-Eos'] = np.nan
    all_series.loc[all_series['Differential-Eos'] > 100 ,   'Differential-Eos'] = np.nan
    all_series.loc[all_series['Differential-Basos'] < 0 ,    'Differential-Basos'] = np.nan
    all_series.loc[all_series['Differential-Basos'] > 15 ,   'Differential-Basos'] = np.nan
    all_series.loc[all_series['Differential-Lymphs'] < 0 ,    'Differential-Lymphs'] = np.nan
    all_series.loc[all_series['Differential-Lymphs'] > 105 ,  'Differential-Lymphs'] = np.nan
    all_series.loc[all_series['Differential-Neuts'] < 0 ,    'Differential-Neuts'] = np.nan
    all_series.loc[all_series['Differential-Neuts'] > 105 ,  'Differential-Neuts'] = np.nan
    all_series.loc[all_series['Differential-Monos'] < 0 ,    'Differential-Monos'] = np.nan
    all_series.loc[all_series['Differential-Monos'] > 100 ,  'Differential-Monos'] = np.nan
    all_series.loc[all_series['Differential-Bands'] < 0 ,    'Differential-Bands'] = np.nan
    all_series.loc[all_series['Differential-Bands'] > 75 ,   'Differential-Bands'] = np.nan
    all_series.loc[all_series['Differential-Polys'] < 0 ,    'Differential-Polys'] = np.nan
    all_series.loc[all_series['Differential-Polys'] > 105 ,  'Differential-Polys'] = np.nan
    all_series.loc[all_series['Absolute Lymphocyte Count'] < 0 ,     'Absolute Lymphocyte Count'] = np.nan
    all_series.loc[all_series['Absolute Lymphocyte Count'] > 3000 ,  'Absolute Lymphocyte Count'] = np.nan
    all_series.loc[all_series['Eosinophil Count'] < 0 ,     'Eosinophil Count'] = np.nan
    all_series.loc[all_series['Eosinophil Count'] > 1500 ,  'Eosinophil Count'] = np.nan
    all_series.loc[all_series['O2 Flow'] < 0 ,    'O2 Flow'] = np.nan
    all_series.loc[all_series['O2 Flow'] > 80 ,   'O2 Flow'] = np.nan
    all_series.loc[all_series['Oxygen'] < 0 ,    'Oxygen'] = np.nan
    all_series.loc[all_series['Oxygen'] > 105 ,  'Oxygen'] = np.nan
    all_series.loc[all_series['Oxygen Saturation'] < 0 ,    'Oxygen Saturation'] = np.nan
    all_series.loc[all_series['Oxygen Saturation'] > 105 ,  'Oxygen Saturation'] = np.nan
    all_series.loc[all_series['Total CO2'] < 0 ,     'Total CO2'] = np.nan
    all_series.loc[all_series['Total CO2'] > 100 ,   'Total CO2'] = np.nan
    all_series.loc[all_series['Albumin'] < 0 ,    'Albumin'] = np.nan
    all_series.loc[all_series['Albumin'] > 10 ,   'Albumin'] = np.nan
    all_series.loc[all_series['Troponin T'] < 0 ,    'Troponin T'] = np.nan
    all_series.loc[all_series['Troponin T'] > 20 ,   'Troponin T'] = np.nan
    all_series.loc[all_series['Troponin I'] < 0 ,    'Troponin I'] = np.nan
    all_series.loc[all_series['Troponin I'] > 40 ,   'Troponin I'] = np.nan
    all_series.loc[all_series['Vancomycin'] < 0 ,    'Vancomycin'] = np.nan
    all_series.loc[all_series['Vancomycin'] > 70 ,   'Vancomycin'] = np.nan
    all_series.loc[all_series['Triglycerides'] < 0 ,     'Triglycerides'] = np.nan
    all_series.loc[all_series['Triglycerides'] > 1800 ,  'Triglycerides'] = np.nan
    all_series.loc[all_series['Fibrinogen'] < 0 ,     'Fibrinogen'] = np.nan
    all_series.loc[all_series['Fibrinogen'] > 1200 ,  'Fibrinogen'] = np.nan
    all_series.loc[all_series['Transferrin'] < 0 ,    'Transferrin'] = np.nan
    all_series.loc[all_series['Transferrin'] > 550 ,  'Transferrin'] = np.nan
    all_series.loc[all_series['Ferritin'] < 0 ,     'Ferritin'] = np.nan
    all_series.loc[all_series['Ferritin'] > 5000 ,  'Ferritin'] = np.nan
    all_series.loc[all_series['Cortisol'] < 0 ,    'Cortisol'] = np.nan
    all_series.loc[all_series['Cortisol'] > 250 ,  'Cortisol'] = np.nan
    all_series.loc[all_series['Protein'] < 0 ,    'Protein'] = np.nan
    all_series.loc[all_series['Protein'] > 700 ,  'Protein'] = np.nan
    all_series.loc[all_series['Total Protein'] < 0 ,    'Total Protein'] = np.nan
    all_series.loc[all_series['Total Protein'] > 25 ,   'Total Protein'] = np.nan
    all_series.loc[all_series['PEEP'] < 0 ,    'PEEP'] = np.nan
    all_series.loc[all_series['PEEP'] > 40 ,   'PEEP'] = np.nan
    all_series.loc[all_series['Tidal Volume'] < 0 ,     'Tidal Volume'] = np.nan
    all_series.loc[all_series['Tidal Volume'] > 1100 ,  'Tidal Volume'] = np.nan
    all_series.loc[all_series['Ventilation Rate'] < 0 ,    'Ventilation Rate'] = np.nan
    all_series.loc[all_series['Ventilation Rate'] > 75 ,   'Ventilation Rate'] = np.nan
    all_series.loc[all_series['Granulocyte Count'] < 0 ,      'Granulocyte Count'] = np.nan
    all_series.loc[all_series['Granulocyte Count'] > 20000 ,  'Granulocyte Count'] = np.nan
    all_series.loc[all_series['Promyelocytes'] < 0 ,    'Promyelocytes'] = np.nan
    all_series.loc[all_series['Promyelocytes'] > 100 ,  'Promyelocytes'] = np.nan
    all_series.loc[all_series['Metamyelocytes'] < 0 ,    'Metamyelocytes'] = np.nan
    all_series.loc[all_series['Metamyelocytes'] > 25 ,   'Metamyelocytes'] = np.nan
    all_series.loc[all_series['Myelocytes'] < 0 ,    'Myelocytes'] = np.nan
    all_series.loc[all_series['Myelocytes'] > 25 ,   'Myelocytes'] = np.nan
    all_series.loc[all_series['Ovalocytes'] < 0 ,    'Ovalocytes'] = np.nan
    all_series.loc[all_series['Ovalocytes'] > 10 ,   'Ovalocytes'] = np.nan
    all_series.loc[all_series['Heart Rate'] < 20 ,    'Heart Rate'] = np.nan
    all_series.loc[all_series['Heart Rate'] > 200 ,   'Heart Rate'] = np.nan
    all_series.loc[all_series['Respiratory Rate'] < 1 ,    'Respiratory Rate'] = np.nan
    all_series.loc[all_series['Respiratory Rate'] > 60 ,   'Respiratory Rate'] = np.nan
    all_series.loc[all_series['Respiratory Rate (Total)'] < 1 ,    'Respiratory Rate (Total)'] = np.nan
    all_series.loc[all_series['Respiratory Rate (Total)'] > 60 ,   'Respiratory Rate (Total)'] = np.nan
    all_series.loc[all_series['Respiratory Rate (Set)'] < 1 ,    'Respiratory Rate (Set)'] = np.nan
    all_series.loc[all_series['Respiratory Rate (Set)'] > 60 ,   'Respiratory Rate (Set)'] = np.nan
    all_series.loc[all_series['Non Invasive Blood Pressure mean'] < 10 ,    'Non Invasive Blood Pressure mean'] = np.nan
    all_series.loc[all_series['Non Invasive Blood Pressure mean'] > 200 ,   'Non Invasive Blood Pressure mean'] = np.nan
    all_series.loc[all_series['Non Invasive Blood Pressure diastolic'] < 10 ,    'Non Invasive Blood Pressure diastolic'] = np.nan
    all_series.loc[all_series['Non Invasive Blood Pressure diastolic'] > 200 ,   'Non Invasive Blood Pressure diastolic'] = np.nan
    all_series.loc[all_series['Non Invasive Blood Pressure systolic'] < 10 ,    'Non Invasive Blood Pressure systolic'] = np.nan
    all_series.loc[all_series['Non Invasive Blood Pressure systolic'] > 250 ,   'Non Invasive Blood Pressure systolic'] = np.nan
    all_series.loc[all_series['Arterial Blood Pressure mean'] < 10 ,    'Arterial Blood Pressure mean'] = np.nan
    all_series.loc[all_series['Arterial Blood Pressure mean'] > 200 ,   'Arterial Blood Pressure mean'] = np.nan
    all_series.loc[all_series['Arterial Blood Pressure diastolic'] < 10 ,    'Arterial Blood Pressure diastolic'] = np.nan
    all_series.loc[all_series['Arterial Blood Pressure diastolic'] > 200 ,   'Arterial Blood Pressure diastolic'] = np.nan
    all_series.loc[all_series['Arterial Blood Pressure systolic'] < 10 ,    'Arterial Blood Pressure systolic'] = np.nan
    all_series.loc[all_series['Arterial Blood Pressure systolic'] > 250 ,   'Arterial Blood Pressure systolic'] = np.nan
    all_series.loc[all_series['Pulmonary Artery Pressure mean'] < 1 ,    'Pulmonary Artery Pressure mean'] = np.nan
    all_series.loc[all_series['Pulmonary Artery Pressure mean'] > 100 ,  'Pulmonary Artery Pressure mean'] = np.nan
    all_series.loc[all_series['Pulmonary Artery Pressure diastolic'] < 1 ,    'Pulmonary Artery Pressure diastolic'] = np.nan
    all_series.loc[all_series['Pulmonary Artery Pressure diastolic'] > 80 ,   'Pulmonary Artery Pressure diastolic'] = np.nan
    all_series.loc[all_series['Pulmonary Artery Pressure systolic'] < 1 ,    'Pulmonary Artery Pressure systolic'] = np.nan
    all_series.loc[all_series['Pulmonary Artery Pressure systolic'] > 140 ,  'Pulmonary Artery Pressure systolic'] = np.nan
    all_series.loc[all_series['Mean Airway Pressure'] < 0 ,    'Mean Airway Pressure'] = np.nan
    all_series.loc[all_series['Mean Airway Pressure'] > 60 ,   'Mean Airway Pressure'] = np.nan
    all_series.loc[all_series['Pain Level'] < 0 ,    'Pain Level'] = np.nan
    all_series.loc[all_series['Pain Level'] > 12 ,   'Pain Level'] = np.nan
    all_series.loc[all_series['Pain Present'] < 0 ,   'Pain Present'] = np.nan
    all_series.loc[all_series['Pain Present'] > 5 ,   'Pain Present'] = np.nan
    all_series.loc[all_series['GCS Total'] < 0 ,    'GCS Total'] = np.nan
    all_series.loc[all_series['GCS Total'] > 20 ,   'GCS Total'] = np.nan
    all_series.loc[all_series['GCS - Eye Opening'] < 0 ,  'GCS - Eye Opening'] = np.nan
    all_series.loc[all_series['GCS - Eye Opening'] > 8 ,  'GCS - Eye Opening'] = np.nan
    all_series.loc[all_series['GCS - Motor Response'] < 0 ,  'GCS - Motor Response'] = np.nan
    all_series.loc[all_series['GCS - Motor Response'] > 8 ,  'GCS - Motor Response'] = np.nan
    all_series.loc[all_series['GCS - Verbal Response'] < 0 ,  'GCS - Verbal Response'] = np.nan
    all_series.loc[all_series['GCS - Verbal Response'] > 8 ,  'GCS - Verbal Response'] = np.nan
    all_series.loc[all_series['Mental status'] < 0 ,    'Mental status'] = np.nan
    all_series.loc[all_series['Mental status'] > 16 ,   'Mental status'] = np.nan
    all_series.loc[all_series['Richmond-RAS Scale'] < -7 ,   'Richmond-RAS Scale'] = np.nan
    all_series.loc[all_series['Richmond-RAS Scale'] > 7 ,    'Richmond-RAS Scale'] = np.nan
    all_series.loc[all_series['Goal Richmond-RAS Scale'] < 0 ,    'Goal Richmond-RAS Scale'] = np.nan
    all_series.loc[all_series['Goal Richmond-RAS Scale'] > 7 ,    'Goal Richmond-RAS Scale'] = np.nan
    all_series.loc[all_series['Risk for Falls'] < 0 ,    'Risk for Falls'] = np.nan
    all_series.loc[all_series['Risk for Falls'] > 5 ,    'Risk for Falls'] = np.nan
    all_series.loc[all_series['Delirium assessment'] < 0 ,    'Delirium assessment'] = np.nan
    all_series.loc[all_series['Delirium assessment'] > 5 ,    'Delirium assessment'] = np.nan
    all_series.loc[all_series['CAM-ICU MS Change'] < 0 ,    'CAM-ICU MS Change'] = np.nan
    all_series.loc[all_series['CAM-ICU MS Change'] > 5 ,    'CAM-ICU MS Change'] = np.nan
    all_series.loc[all_series['CAM-ICU RASS LOC'] < 0 ,    'CAM-ICU RASS LOC'] = np.nan
    all_series.loc[all_series['CAM-ICU RASS LOC'] > 5 ,    'CAM-ICU RASS LOC'] = np.nan
    all_series.loc[all_series['CAM-ICU Inattention'] < 0 ,    'CAM-ICU Inattention'] = np.nan
    all_series.loc[all_series['CAM-ICU Inattention'] > 5 ,    'CAM-ICU Inattention'] = np.nan
    all_series.loc[all_series['CAM-ICU Altered LOC'] < 0 ,    'CAM-ICU Altered LOC'] = np.nan
    all_series.loc[all_series['CAM-ICU Altered LOC'] > 5 ,    'CAM-ICU Altered LOC'] = np.nan
    all_series.loc[all_series['CAM-ICU Disorganized thinking'] < 0 ,    'CAM-ICU Disorganized thinking'] = np.nan
    all_series.loc[all_series['CAM-ICU Disorganized thinking'] > 110 ,  'CAM-ICU Disorganized thinking'] = np.nan
    all_series.loc[all_series['Flow Rate (L/min)'] < 0 ,    'Flow Rate (L/min)'] = np.nan
    all_series.loc[all_series['Flow Rate (L/min)'] > 180 ,  'Flow Rate (L/min)'] = np.nan
    all_series.loc[all_series['FiO2'] < 0 ,    'FiO2'] = np.nan
    all_series.loc[all_series['FiO2'] > 105 ,  'FiO2'] = np.nan
    all_series.loc[all_series['FiO2 (Set)'] < 0 ,    'FiO2 (Set)'] = np.nan
    all_series.loc[all_series['FiO2 (Set)'] > 105 ,  'FiO2 (Set)'] = np.nan
    all_series.loc[all_series['SpO2'] < 0 ,    'SpO2'] = np.nan
    all_series.loc[all_series['SpO2'] > 105 ,  'SpO2'] = np.nan
    all_series.loc[all_series['SvO2'] < 0 ,    'SvO2'] = np.nan
    all_series.loc[all_series['SvO2'] > 105 ,  'SvO2'] = np.nan
    all_series.loc[all_series['Admission Weight (Kg)'] < 0 ,    'Admission Weight (Kg)'] = np.nan
    all_series.loc[all_series['Admission Weight (Kg)'] > 260 ,  'Admission Weight (Kg)'] = np.nan
    all_series.loc[all_series['Height (cm)'] < 0 ,    'Height (cm)'] = np.nan
    all_series.loc[all_series['Height (cm)'] > 215 ,  'Height (cm)'] = np.nan
    all_series.loc[all_series['CVP'] < 0 ,    'CVP'] = np.nan
    all_series.loc[all_series['CVP'] > 80 ,   'CVP'] = np.nan
    all_series.loc[all_series['ETCO2'] < 0 ,    'ETCO2'] = np.nan
    all_series.loc[all_series['ETCO2'] > 110 ,  'ETCO2'] = np.nan
    all_series.loc[all_series['Calcium non-ionized'] < 0 ,    'Calcium non-ionized'] = np.nan
    all_series.loc[all_series['Calcium non-ionized'] > 20 ,   'Calcium non-ionized'] = np.nan
    all_series.loc[all_series['Plateau Pressure'] < 0 ,    'Plateau Pressure'] = np.nan
    all_series.loc[all_series['Plateau Pressure'] > 80 ,  'Plateau Pressure'] = np.nan
    all_series.loc[all_series['Vancomycin (Trough)'] < 0 ,    'Vancomycin (Trough)'] = np.nan
    all_series.loc[all_series['Vancomycin (Trough)'] > 70 ,   'Vancomycin (Trough)'] = np.nan
    all_series.loc[all_series['Vancomycin (Peak)'] < 0 ,    'Vancomycin (Peak)'] = np.nan
    all_series.loc[all_series['Vancomycin (Peak)'] > 70 ,   'Vancomycin (Peak)'] = np.nan
    all_series.loc[all_series['Vancomycin (Random)'] < 0 ,    'Vancomycin (Random)'] = np.nan
    all_series.loc[all_series['Vancomycin (Random)'] > 70 ,   'Vancomycin (Random)'] = np.nan
    all_series.loc[all_series['PEEP (Set)'] < 0 ,    'PEEP (Set)'] = np.nan
    all_series.loc[all_series['PEEP (Set)'] > 40 ,   'PEEP (Set)'] = np.nan
    all_series.loc[all_series['Tidal Volume (Set)'] < 0 ,     'Tidal Volume (Set)'] = np.nan
    all_series.loc[all_series['Tidal Volume (Set)'] > 1000 ,  'Tidal Volume (Set)'] = np.nan
    all_series.loc[all_series['Pressure Support'] < 0 ,    'Pressure Support'] = np.nan
    all_series.loc[all_series['Pressure Support'] > 70 ,   'Pressure Support'] = np.nan

    return all_series

### Convert data to timeseries

In [7]:
def convert_events_to_timeseries(all_tables, all_variables):
    
    metadata  = all_tables[['CHARTTIME', 'ICUSTAY_ID']].sort_values(by=['CHARTTIME'])\
                    .drop_duplicates(keep='first').set_index('CHARTTIME')

    timeserie = all_tables[['CHARTTIME', 'LABEL', 'VALUE']]\
                    .sort_values(by=['CHARTTIME'], axis=0)\
                    .drop_duplicates(subset=['CHARTTIME', 'LABEL'], keep='last')

    time_piv   = timeserie.pivot(index='CHARTTIME', columns='LABEL', values='VALUE')
    timeseries = time_piv.merge(metadata, left_index=True, right_index=True).sort_index(axis=0).reset_index()
    
    missing_vars = [v for v in all_variables if v not in timeseries.columns]
    missing_dataframes = [pd.DataFrame({v: np.nan}, index=timeseries.index) for v in missing_vars]

    if missing_dataframes:
        timeseries = pd.concat([timeseries] + missing_dataframes, axis=1)
                    
    return timeseries

### Fix varibales

In [8]:
def convert_variables(df):

    df_copy = df.copy()

    df_copy['Admission Weight (Kg)'] = df_copy.groupby("ICUSTAY_ID")['Admission Weight (Kg)'].transform(lambda x: x.fillna(x.mean())).mean()
    df_copy['Height (cm)'] = df_copy.groupby("ICUSTAY_ID")['Height (cm)'].transform(lambda x: x.fillna(x.mean())).mean()
    df_copy['Admission Weight (Kg)'] = df_copy['Admission Weight (Kg)'].round(1)
    df_copy['Height (cm)'] = df_copy['Height (cm)'].round(1)

    df_copy.loc[df_copy['GCS Total'].isnull(), 'GCS Total'] = df_copy['GCS - Eye Opening'] + df_copy['GCS - Motor Response'] + df_copy['GCS - Verbal Response']

    df_copy.rename(columns={'Admission Weight (Kg)': 'Weight', 'Height (cm)': 'Height'}, inplace=True)
    
    return df_copy

### Seperate Vital Sign Variables

In [9]:
def vital_sign_frame(df):
    
    vital_columns = ['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'GENDER', 'AGE', 'ETHNICITY', 'CHARTTIME', 'INTIME',
                     'Heart Rate', 'Temperature', 'Respiratory Rate', 'SpO2', 'Oxygen Saturation',
                     'Non Invasive Blood Pressure mean', 'Non Invasive Blood Pressure diastolic', 
                     'Non Invasive Blood Pressure systolic' , 'Arterial Blood Pressure mean',
                     'Arterial Blood Pressure diastolic', 'Arterial Blood Pressure systolic']
    
    vital_df = df[vital_columns]
    vital_df = vital_df.drop_duplicates()
    vital_df = vital_df.dropna(thresh=9, axis=0)
    vital_df = vital_df.reset_index(drop=True).sort_values(by= 'CHARTTIME')
    
    return vital_df

### Create hourly based data frames

In [10]:
def create_template_all_frame(all_series):
    
    id_los = all_series.groupby('ICUSTAY_ID')[['ICUSTAY_ID','ICU_LOS_H']].head(1)
    id_los['ICU_LOS_H'] = id_los['ICU_LOS_H'].apply(np.ceil) + 1

    stayids = list(id_los.ICUSTAY_ID.unique())
    template_list = []

    for id in stayids:
        df = {'Bins': range(0, 1 + id_los[id_los.ICUSTAY_ID == id].ICU_LOS_H.values[0].astype(int)) }
        df = pd.DataFrame(df)
        df['ICUSTAY_ID'] = id
        template_list.append(df)

    template_df = pd.concat(template_list)
    
    return template_df

In [11]:
def create_template_vital_frame(all_series, bt_vital=30):
    
    id_los = all_series.groupby('ICUSTAY_ID')[['ICUSTAY_ID', 'ICU_LOS_H']].head(1)
    id_los['ICU_LOS_H']  = (id_los['ICU_LOS_H']).apply(np.ceil) + 1

    stayids = list(id_los.ICUSTAY_ID.unique())
    template_list = []

    for id in stayids:
        df = {'Bins': range(0, (1 + id_los[id_los.ICUSTAY_ID == id].ICU_LOS_H.values[0].astype(int)) * 60, bt_vital) }
        df = pd.DataFrame(df)
        df['ICUSTAY_ID'] = id
        template_list.append(df)

    template_df = pd.concat(template_list)
    
    return template_df

### Calculate bins 

In [12]:
def bin_time(frame_df, frame_vital, bt_df=60, bt_vital=30):

    frame_df.CHARTTIME = pd.to_datetime(frame_df.CHARTTIME)
    frame_df.INTIME = pd.to_datetime(frame_df.INTIME)
    
    frame_vital.CHARTTIME = pd.to_datetime(frame_vital.CHARTTIME)
    frame_vital.INTIME = pd.to_datetime(frame_vital.INTIME)
    
    frame_df['Hours'] = ((frame_df.CHARTTIME - frame_df.INTIME).dt.total_seconds()) / 60./60
    frame_df = frame_df[frame_df['Hours'] > 0].copy()    
    frame_df['Minutes'] = frame_df['Hours'].apply(lambda s: s * 60)
    
    frame_vital['Hours'] = ((frame_vital.CHARTTIME - frame_vital.INTIME).dt.total_seconds()) / 60./60
    frame_vital = frame_vital[frame_vital['Hours'] > 0].copy()    
    frame_vital['Minutes'] = frame_vital['Hours'].apply(lambda s: s * 60)
    
    frame_df['Bins'] = (frame_df['Minutes'] / bt_df).astype(int)
    frame_df.drop(columns=['Hours', 'Minutes'], inplace=True)
    
    frame_vital['Bins'] = (frame_vital['Minutes'] / bt_vital).astype(int) * bt_vital
    frame_vital.drop(columns=['Hours', 'Minutes'], inplace=True)
    
    return frame_df, frame_vital

### Seperating categorical, continuse, static variables

In [13]:
def seperate_varibale_types(all_series):
    
    static_columns = ['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'ADMITTIME', 'DISCHTIME', 'INTIME', 'OUTTIME', 
                      'EXPIRE_FLAG', 'GENDER', 'AGE', 'ETHNICITY', 'ICU_LOS_H', 'ICU_LOS_D', 'HOSP_LOS_H', 
                      'HOSP_LOS_D', 'ICU_EXPIRE_FLAG', 'HOSPITAL_EXPIRE_FLAG', 'DEATH_TIME_DISCH', 
                      'Weight', 'Height']
    
    categorical_columns = ['ICUSTAY_ID', 'Bins', 'Microbio Test', 'Blood Culture', 'Note', 'Discharge_Note', 
                           '22 Gauge Insertion Date', '20 Gauge Insertion Date', '18 Gauge Insertion Date',
                           'Arterial line Insertion Date', 'Arterial line Tubing Change', 
                           'Arterial Line Dressing Change', 'Multi Lumen Insertion Date','Multi Lumen Cap Change',
                           'Multi Lumen Dressing Change', 'Multi Lumen Tubing Change',
                           'Antibiotic_PRC', 'Fentanyl_PRC', 'Propofol_PRC', 'Norepinephrine_PRC', 'Insulin_PRC', 
                           'Midazolam_PRC', 'Heparin_PRC', 'Dexmedetomidine_PRC', 'Amiodarone_PRC', 
                           'Vasopressin_PRC', 'Phenylephrine_PRC', 'Dopamine_PRC', 'Nicardipine_PRC', 
                           'Milrinone_PRC', 'Pantoprazole_PRC', 'Diltiazem_PRC', 'Dobutamine_PRC', 
                           'Nitroglycerin_PRC', 'Epinephrine_PRC', 'Warfarin_PRC', 'Apixaban_PRC', 
                           'Dabigatran_PRC', 'Rivaroxaban_PRC', 'Edoxaban_PRC',
                           'Intubated', 'Skin Temperature', 'Pain Level', 'Pain Present', 'GCS Total', 
                           'GCS - Eye Opening', 'GCS - Motor Response' , 'GCS - Verbal Response', 
                           'Mental status', 'Richmond-RAS Scale', 'Goal Richmond-RAS Scale', 'Risk for Falls', 
                           'Delirium assessment', 'CAM-ICU MS Change', 'CAM-ICU RASS LOC', 'CAM-ICU Inattention',
                           'CAM-ICU Altered LOC', 'CAM-ICU Disorganized thinking',
                           'Heart Rhythm', 'Ventilator Mode', 'Ventilator Type', 'Ventilator',
                           'Sedation Score', 'Sedation Scale', 'EtCO2 Clinical indication']

    all_columns = list(all_series.columns)
    continuse_columns = (set(all_columns) - set(categorical_columns)) - set(static_columns)
    continuse_columns = list(continuse_columns)
    continuse_columns.remove('CHARTTIME')
    continuse_columns.extend(['ICUSTAY_ID', 'Bins'])
    
    return static_columns, categorical_columns, continuse_columns

### Seperate dataframes based on variables type

In [14]:
def seperate_frame_types(all_series, static_columns, categorical_columns, continuse_columns):
    
    static_df = all_series[static_columns]
    static_df = static_df.drop_duplicates()

    categorical_df = all_series[categorical_columns]
    categorical_df = categorical_df.drop_duplicates()
    categorical_df = categorical_df.dropna(thresh=3, axis=0)

    dynamic_df = all_series[continuse_columns]
    dynamic_df = dynamic_df.drop_duplicates()
    dynamic_df = dynamic_df.dropna(thresh=3, axis=0)
    
    return static_df, categorical_df, dynamic_df

### Binning data

In [15]:
def binning(template_df, categorical_df, dynamic_df, categorical_columns, continuse_columns):
    
    cat_columns = [col for col in categorical_columns if col not in ('ICUSTAY_ID', 'Bins')]
    cont_column = [col for col in continuse_columns   if col not in ('ICUSTAY_ID', 'Bins')]

    for category in cat_columns:
        temp_df = categorical_df.groupby(['ICUSTAY_ID', 'Bins'], group_keys=True)[category].apply(pd.Series.mode).reset_index()[['ICUSTAY_ID', 'Bins', category]]
        if category not in ['Note', 'Discharge_Note']:
            temp_df.drop_duplicates(['ICUSTAY_ID', 'Bins'], inplace=True)
        template_df = pd.merge(template_df, temp_df, on=['ICUSTAY_ID', 'Bins'], how='left')

    for continuse in cont_column:
        temp_df = dynamic_df.groupby(['ICUSTAY_ID', 'Bins'], group_keys=True)[continuse].apply(pd.Series.mean).reset_index()[['ICUSTAY_ID', 'Bins', continuse]]
        temp_df[continuse] = round(temp_df[continuse], 1)
        template_df = pd.merge(template_df, temp_df, on=['ICUSTAY_ID', 'Bins'], how='left')
        
    return template_df

In [16]:
def binning_vital(template_vital_df, df_bin_vital):

    cont_column = ['Heart Rate', 'Temperature', 'Respiratory Rate', 'SpO2', 'Oxygen Saturation',
                   'Non Invasive Blood Pressure mean', 'Non Invasive Blood Pressure diastolic', 
                   'Non Invasive Blood Pressure systolic' , 'Arterial Blood Pressure mean',
                   'Arterial Blood Pressure diastolic', 'Arterial Blood Pressure systolic']

    for continuse in cont_column:
        temp_df = df_bin_vital.groupby(['ICUSTAY_ID', 'Bins'], group_keys=True)[continuse].apply(pd.Series.mean).reset_index()[['ICUSTAY_ID', 'Bins', continuse]]
        temp_df[continuse] = round(temp_df[continuse], 1)
        template_vital_df = pd.merge(template_vital_df, temp_df, on=['ICUSTAY_ID', 'Bins'], how='left')
        
    return template_vital_df

In [17]:
VentilatorType = {
                'Drager': ['Drager', '1.0', 1.0],
                '7200A' : ['7200A', '7200.0', 7200.0],    
                'Servo 900c': ['Servo 900c', '900.0', 900.0],
                'Other/Remarks':['Other/Remarks'],
                'Avea': ['Avea', '2.0', 2.0],
                'Other': ['Other', '6.0', 6.0],
                'Respironics BiPAP':['Respironics BiPAP', '4.0', 4.0],
                'BiPAP/CPAP': ['BiPAP/CPAP', '0.0', 0.0],
                'Sensor Medic (HFO)': ['Sensor Medic (HFO)', '7.0', 7.0],
                'PB 7200': ['PB 7200', '5.0', 5.0]
                }

VentilatorMode = {
                'SIMV+PS': ['SIMV+PS'],
                'Assist Control': ['Assist Control'],
                'SIMV': ['SIMV', '6.0', 6.0],
                'CPAP+PS': ['CPAP+PS'],
                'Pressure Support': ['Pressure Support'],
                'Other/Remarks': ['Other/Remarks'],
                'Pressure Control': ['Pressure Control'],
                'CMV': ['CMV', '1.0', 1.0],
                'TCPCV': ['TCPCV'],
                'CPAP': ['CPAP', '10.0', 10.0],
                'PCV+Assist': ['PCV+Assist', '71.0', 71.0],
                'CMV/ASSIST/AutoFlow': ['CMV/ASSIST/AutoFlow', '49.0', 49.0],
                'CPAP/PSV': ['CPAP/PSV', '11.0', 11.0],
                'CMV/ASSIST': ['CMV/ASSIST', '2.0', 2.0],
                'CMV/AutoFlow': ['CMV/AutoFlow', '48.0', 48.0],
                'Standby': ['Standby', '30.0', 30.0],
                'MMV/PSV/AutoFlow': ['MMV/PSV/AutoFlow', '51.0', 51.0],
                'MMV/PSV': ['MMV/PSV', '13.0', 13.0],
                'CPAP/PPS': ['CPAP/PPS', '53.0', 53.0],
                'MMV': ['MMV', '12.0', 12.0],
                'SIMV/PSV/AutoFlow': ['SIMV/PSV/AutoFlow', '47.0', 47.0],
                'SIMV/PSV': ['SIMV/PSV', '7.0', 7.0],
                'SIMV/AutoFlow': ['SIMV/AutoFlow', '46.0', 46.0],
                'APRV': ['APRV', '26.0', 26.0],
                'PCV+': ['PCV+', '14.0', 14.0],
                'PCV+/PSV': ['PCV+/PSV', '45.0', 45.0],
                'Apnea Ventilation': ['Apnea Ventilation', '17.0', 17.0],
                'SYNCHRON MASTER': ['SYNCHRON MASTER', '15.0', 15.0],
                'MMV/AutoFlow': ['MMV/AutoFlow', '50.0', 50.0],
                'SYNCHRON SLAVE': ['SYNCHRON SLAVE', '16.0', 16.0],
                'Others': ['0.0', 0.0, 'PSV/SBT', 'PRVC/AC', 'CPAP/PSV+ApnVol', 'VOL/AC', 'CPAP/PSV+ApnPres',
                           'CPAP/PSV+Apn TCPL', 'SIMV/VOL', 'PRES/AC', 'APRV/Biphasic+ApnVol', 'PRVC/SIMV',
                           'APRV/Biphasic+ApnPress', 'SIMV/PRES', 'ASYNCHRON MASTER']
                }

ventilator_type_mapping = {v: k for k, values in VentilatorType.items() for v in values}
ventilator_mode_mapping = {v: k for k, values in VentilatorMode.items() for v in values}

In [18]:
def convert_ventilator_column(df):
    
    df_copy = df.copy()
    
    mask = df_copy['Intubated'].notna()
    df_copy.loc[mask, 'Intubated'] = df_copy.loc[mask, 'Intubated'].astype(float).astype(int)
    df_copy["Ventilator Mode"] = df_copy["Ventilator Mode"].map(ventilator_mode_mapping)
    df_copy["Ventilator Type"] = df_copy["Ventilator Type"].map(ventilator_type_mapping)
    
    return df_copy

### Extract timeseries data based on ICU - multiprocessing

In [19]:
def process_csv(stay_dir):
    
    dn = os.path.join(path_timeseries, stay_dir)
        
    try:
        sys.stdout.flush()
        
        admission  = dataframe_from_csv(os.path.join(path_timeseries, stay_dir, 'admission.csv'))
        all_tables = dataframe_from_csv(os.path.join(path_timeseries, stay_dir, 'all_tables.csv'))

        all_tables = filter_on_variabels(all_tables, all_variables)
        all_tables = all_tables.drop_duplicates()
        all_tables = all_tables.sort_values(by=['CHARTTIME'])

        timeepisode = convert_events_to_timeseries(all_tables, all_variables)
        timeepisode = outlier_removal(timeepisode)

        merged_df = pd.merge(timeepisode, admission, on='ICUSTAY_ID')
        merged_df = merged_df.sort_values(by=['CHARTTIME'])
        merged_df = convert_variables(merged_df)

        vital_df = vital_sign_frame(merged_df)

        template_df = create_template_all_frame(merged_df)
        template_vital_df = create_template_vital_frame(merged_df)
        df_bin, df_bin_vital = bin_time(merged_df, vital_df)

        static_columns, categorical_columns, continuse_columns = seperate_varibale_types(df_bin)
        static_df, categorical_df, dynamic_df = seperate_frame_types(df_bin, static_columns, categorical_columns, continuse_columns)

        binned_df = binning(template_df, categorical_df, dynamic_df, categorical_columns, continuse_columns)
        binned_vital_df = binning_vital(template_vital_df, df_bin_vital)
        
        binned_df = convert_ventilator_column(binned_df)

        final = pd.merge(binned_df, static_df, on='ICUSTAY_ID', how='left')
        
        final = final.reset_index(drop=True)
        binned_vital_df = binned_vital_df.reset_index(drop=True)

        final.to_csv(os.path.join(dn, 'raw_timeseries.csv'), index=False)
        binned_vital_df.to_csv(os.path.join(dn, 'raw_vital_timeseries.csv'), index=False)
        
    except Exception as e:
        print(f"Error processing {stay_dir}: {e}")
        exception_stayID.append(stay_dir)

In [22]:
dirs = os.listdir(path_timeseries)

exception_stayID = []
num_processes = cpu_count()

In [ ]:
with Pool(num_processes) as p:
    for _ in tqdm(p.imap(process_csv, dirs), total=len(dirs)):
        pass

In [26]:
if exception_stayID:
    with open("../Data/Cohort/exception_stayID.txt", "w") as f:
        for subject_id in exception_stayID:
            f.write(str(subject_id) +"\n")

### Extract timeseries data based on ICU - For Loop

In [17]:
def extract_time_series_from_subject(output_path, all_variables):
    
    dirs = os.listdir(output_path)
    total_dirs = len(dirs)
    processed_dirs = 0
    last_printed_progress = -1
    
    exception_stayID = []
    
    for stay_dir in dirs:
        dn = os.path.join(output_path, stay_dir)
        
        try:
            sys.stdout.flush()

            admission  = dataframe_from_csv(os.path.join(output_path, stay_dir, 'admission.csv'))
            all_tables = dataframe_from_csv(os.path.join(output_path, stay_dir, 'all_tables.csv'))
            
            all_tables = filter_on_variabels(all_tables, all_variables)
            all_tables = all_tables.drop_duplicates()
            all_tables = all_tables.sort_values(by=['CHARTTIME'])

            timeepisode = convert_events_to_timeseries(all_tables, all_variables)
            timeepisode = outlier_removal(timeepisode)

            merged_df = pd.merge(timeepisode, admission, on='ICUSTAY_ID')
            merged_df = merged_df.sort_values(by=['CHARTTIME'])
            merged_df = convert_variables(merged_df)

            vital_df = vital_sign_frame(merged_df)

            template_df = create_template_all_frame(merged_df)
            template_vital_df = create_template_vital_frame(merged_df)
            df_bin, df_bin_vital = bin_time(merged_df, vital_df)

            static_columns, categorical_columns, continuse_columns = seperate_varibale_types(df_bin)
            static_df, categorical_df, dynamic_df = seperate_frame_types(df_bin, static_columns, categorical_columns, continuse_columns)

            binned_df = binning(template_df, categorical_df, dynamic_df, categorical_columns, continuse_columns)
            binned_vital_df = binning_vital(template_vital_df, df_bin_vital)

            final = pd.merge(binned_df, static_df, on='ICUSTAY_ID', how='left')
            final = final.reset_index(drop=True)
            binned_vital_df = binned_vital_df.reset_index(drop=True)

            final.to_csv(os.path.join(dn, 'raw_timeseries.csv'), index=False)
            binned_vital_df.to_csv(os.path.join(dn, 'raw_vital_timeseries.csv'), index=False)
            
            processed_dirs += 1
            progress_percentage = (processed_dirs / total_dirs) * 100
            
            if int(progress_percentage) > last_printed_progress:
                last_printed_progress = int(progress_percentage)
                sys.stdout.write(f"\rProgress: {progress_percentage:.2f}% ({processed_dirs}/{total_dirs}) - Processing StayID {stay_dir}...\n")
                sys.stdout.flush()
                
        except :
            print(stay_dir)
            exception_stayID.append(stay_dir)
            continue
            
    print('DONE')
    
    with open("../Data/Cohort/exception_stayID.txt", "w") as f:
        for subject_id in exception_stayID:
            f.write(str(subject_id) +"\n")

In [1]:
# extract_time_series_from_subject(path_timeseries, all_variables)